In [1]:
import sys
sys.path.insert(0, '../..')

In [ ]:
import os
import csv
import cv2
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from courthawk_engine.pose_estimation import PoseEstimator

DATA_DIR = '../data/pose_estimation_data'
METADATA_PATH = os.path.join(DATA_DIR, 'shot_classification_metadata.csv')
MODEL_PATH = '../models/shot_classifier_log_regression.pkl'
RF_MODEL_PATH = '../models/shot_classifier_random_forest.pkl'
CLASSES = ['backhand', 'forehand', 'serve']

## Extract keypoints from labeled images

In [3]:
estimator = PoseEstimator('../models/pose_landmarker.task')

X, y = [], []
no_pose = 0
unlabeled = 0

with open(METADATA_PATH, newline = '') as f:
    reader = csv.reader(f)
    next(reader)  # header row

    for row in reader:
        image_path, shot_type = row[0].strip(), row[1].strip()

        if shot_type not in CLASSES:
            unlabeled += 1
            continue

        frame = cv2.imread(os.path.join(DATA_DIR, image_path))
        keypoints = estimator.get_keypoints(frame)
        if keypoints is None:
            # Don't delete the image, just skip it and move on
            no_pose += 1
            continue

        X.append(keypoints)
        y.append(CLASSES.index(shot_type))

X = np.array(X)
y = np.array(y)
print(f'Samples: {len(X)}  |  No pose detected (skipped): {no_pose}  |  Unlabeled (skipped): {unlabeled}')
print(f'Class counts: { {CLASSES[i]: int((y == i).sum()) for i in range(len(CLASSES))} }')

I0000 00:00:1786827822.567430 45616472 init-domain.cc:132] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1786827822.651399 45616472 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786827822.727364 45616475 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786827822.740364 45616475 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786827822.903310 45616481 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Samples: 351  |  No pose detected (skipped): 7  |  Unlabeled (skipped): 0
Class counts: {'backhand': 172, 'forehand': 146, 'serve': 33}


## Train Logistic Regression
No testing set only train.

In [4]:
model = LogisticRegression(max_iter = 1500)
model.fit(X, y)

y_pred = model.predict(X)
print(classification_report(y, y_pred, target_names = CLASSES))

              precision    recall  f1-score   support

    backhand       0.90      0.92      0.91       172
    forehand       0.90      0.90      0.90       146
       serve       1.00      0.91      0.95        33

    accuracy                           0.91       351
   macro avg       0.93      0.91      0.92       351
weighted avg       0.91      0.91      0.91       351



## Save model

In [5]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
print(f'Saved to {MODEL_PATH}')

Saved to ../models/shot_classifier_log_regression.pkl


## Train Random Forest
No testing set only train. Uses the same X, y extracted above.

In [6]:
rf_model = RandomForestClassifier(n_estimators = 200, random_state = 42, max_depth = 4, min_samples_leaf = 5)
rf_model.fit(X, y)

rf_y_pred = rf_model.predict(X)
print(classification_report(y, rf_y_pred, target_names = CLASSES))

              precision    recall  f1-score   support

    backhand       0.94      0.94      0.94       172
    forehand       0.93      0.95      0.94       146
       serve       1.00      0.91      0.95        33

    accuracy                           0.94       351
   macro avg       0.96      0.93      0.94       351
weighted avg       0.94      0.94      0.94       351



## Save model

In [ ]:
with open(RF_MODEL_PATH, 'wb') as f:
    pickle.dump(rf_model, f)
print(f'Saved to {RF_MODEL_PATH}')